## **Transformação de Variáveis Categóricas**

* Label Encoder
* Ond-Hot Encoder

In [1]:
import pandas as pd

In [21]:
data = {'Empregador':['Corpo de Bombeiros', 'Secretaria da Saúde', 'Polícia Civil', 'Secretaria da Saúde', 'Polícia Civil'],
        'Idade':[44, 34, 46, 35, 23],
        'Remuneração':[72000, 65000, 98000, 45000, 34000]}

dados = pd.DataFrame(data)
dados.head()

,Empregador,Idade,Remuneração
0,Corpo de Bombeiros,44,72000
1,Secretaria da Saúde,34,65000
2,Polícia Civil,46,98000
3,Secretaria da Saúde,35,45000
4,Polícia Civil,23,34000


In [3]:
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Empregador   5 non-null      object
 1   Idade        5 non-null      int64 
 2   Remuneração  5 non-null      int64 
dtypes: int64(2), object(1)
memory usage: 252.0+ bytes


### **LabelEncoder**

In [4]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

dados['Empregador'] = label_encoder.fit_transform(dados['Empregador'])
dados.head()

,Empregador,Idade,Remuneração
0,0,44,72000
1,2,34,65000
2,1,46,98000
3,2,35,45000
4,1,23,34000


## **E se a Ordem for Importante ?**

### **One-Hot Encoder (dummy)**

In [6]:
data = {'Empregador':['Corpo de Bombeiros', 'Secretaria da Saúde', 'Polícia Civil', 'Secretaria da Saúde', 'Polícia Civil'],
        'Idade':[44, 34, 46, 35, 23],
        'Remuneração':[72000, 65000, 98000, 45000, 34000]}

dados = pd.DataFrame(data)
dados.head()

,Empregador,Idade,Remuneração
0,Corpo de Bombeiros,44,72000
1,Secretaria da Saúde,34,65000
2,Polícia Civil,46,98000
3,Secretaria da Saúde,35,45000
4,Polícia Civil,23,34000


In [10]:
from sklearn.preprocessing import OneHotEncoder

one_hot_encoder = OneHotEncoder()

X = one_hot_encoder.fit_transform(dados['Empregador'].values.reshape(-1, 1)).toarray()
X

array([[1., 0., 0.],
       [0., 0., 1.],
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 1., 0.]])

In [11]:
# Adicionar no Dataframe original
dfOneHot = pd.DataFrame(X, columns=['Empregador_' + str(int(i)) for i in range(dados.shape[1])])
dados = pd.concat([dados, dfOneHot], axis=1)

# Eliminando a columa Empregados
dados.drop(columns=['Empregador'], axis=1, inplace=True)

dados.head()

,Idade,Remuneração,Empregador_0,Empregador_1,Empregador_2
0,44,72000,1.0,0.0,0.0
1,34,65000,0.0,0.0,1.0
2,46,98000,0.0,1.0,0.0
3,35,45000,0.0,0.0,1.0
4,23,34000,0.0,1.0,0.0


**O problema é que pode gerar Multicolinearidade então uma das variáveis Dummy deve ser descartada**

## **(VIF) - Variance Inflation Factor**

* VIF = 1, menos multicolinearidade.
* VIF < 5, multicolinearidade moderada.
* VIF > 5, multicolinearidade extrema (isso que queremos evitar)

In [17]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(data):
    """
    Calcula o Fator de Inflação da Variância (VIF) para cada variável no DataFrame.
    
    Parâmetros:
    - data: DataFrame contendo apenas as variáveis independentes.
    
    Retorna:
    - DataFrame com as variáveis e seus respectivos valores de VIF.
    """
    vif_data = pd.DataFrame()
    vif_data["Variable"] = data.columns
    vif_data["VIF"] = [variance_inflation_factor(data.values, i) for i in range(data.shape[1])]
    
    return vif_data

In [18]:
X = dados.drop(columns=['Remuneração'], axis=1)
calculate_vif(X)

,Variable,VIF
0,Idade,1.272453
1,Empregador_0,8.305660
2,Empregador_1,9.983019
3,Empregador_2,9.983019


### **Eliminar uma variável Dummy**

In [19]:
X = dados.drop(columns=['Remuneração', 'Empregador_2'], axis=1)
calculate_vif(X)

,Variable,VIF
0,Idade,2.631639
1,Empregador_0,1.731809
2,Empregador_1,1.899830


## **Category Enconders**

**1. BinaryEncoder**\
Converte categorias em uma representação binária.
Útil para conjuntos com muitas categorias.

**2. BaseNEncoder**\
Converte categorias em representações numéricas em bases personalizadas (por exemplo, base 2, base 3).
Generalização do BinaryEncoder.

**3. CatBoostEncoder**\
Codificação baseada em estatísticas do target, semelhante ao TargetEncoder.
Inspirado pelo algoritmo CatBoost.

**4. HashingEncoder**\
Aplica o hash para reduzir o número de categorias.
Bom para lidar com alto cardinalidade.

**5. HelmertEncoder**\
Cria contrastes para variáveis categóricas, úteis em modelos estatísticos.

**6. JamesSteinEncoder**\
Usa uma técnica Bayesiana para suavizar a codificação por target.

**7. LeaveOneOutEncoder**\
Codifica uma categoria com base na média do target, excluindo a linha atual.
Reduz o risco de sobreajuste.

**8. MEstimateEncoder**\
Semelhante ao TargetEncoder, mas aplica suavização Bayesiana para evitar viés.

**9. OneHotEncoder**\
Codificação one-hot clássica.
Gera colunas binárias para cada categoria.

**10. OrdinalEncoder**\
Codifica categorias como números inteiros baseados na ordem.
Simples e eficiente para árvores de decisão.

**11. PolynomialEncoder**\
Similar ao HelmertEncoder, mas representa contrastes diferentes.

**12. SumEncoder**\
Cria codificações baseadas em somas, úteis em modelos estatísticos.

**13. TargetEncoder**\
Substitui categorias pela média do target.
Popular em problemas supervisionados.

**14. WeightOfEvidenceEncoder**\
Usa a técnica de Evidence Weighting, comum em modelos de risco de crédito.

**15. BackwardDifferenceEncoder**\
Codifica categorias usando diferenças em relação à média das categorias anteriores.

In [22]:
import category_encoders as ce